1. 🔧 Preparación del entorno

2. 📦 Importación de librerías

In [159]:
import pandas as pd
import numpy as np
import os

3. 📂 Carga de datos

In [160]:
# Cargar los CSV desde rutas absolutas
# Simular la ruta absoluta '/workspace/' desde la raíz del proyecto
try:
    workspace_path = os.path.abspath('workspace')

    sales_df = pd.read_csv(os.path.join(workspace_path, 'sales.csv'))
    inventories_df = pd.read_csv(os.path.join(workspace_path, 'inventories.csv'))
    satisfaction_df = pd.read_csv(os.path.join(workspace_path, 'satisfaction.csv'))
except FileNotFoundError as fe:
    print(f'Error to load the file: {fe}')

In [161]:
# Eliminar filas con valores nulos
sales_df.dropna(inplace=True)
inventories_df.dropna(inplace=True)
satisfaction_df.dropna(inplace=True)

4. 🔍 Exploración de datos

In [162]:
# Ventas totales por producto y por tienda
ventas_por_producto = sales_df.groupby('Producto')['Cantidad_Vendida'].sum()
ventas_por_tienda = sales_df.groupby('ID_Tienda')['Cantidad_Vendida'].sum()

# Ingresos totales por tienda
sales_df['Ingreso'] = sales_df['Cantidad_Vendida'] * sales_df['Precio_Unitario']
ingresos_por_tienda = sales_df.groupby('ID_Tienda')['Ingreso'].sum()

# Resumen estadístico
resumen_ventas = sales_df['Ingreso'].describe()

# Promedio de ventas por tienda y categoría (si existe columna 'Categoría')
if 'Categoría' in sales_df.columns:
    promedio_por_categoria = sales_df.groupby(['ID_Tienda', 'Categoría'])['Cantidad_Vendida'].mean()

5. 📦 Análisis de inventarios

In [163]:
# Calcular ventas totales por producto para unir con inventarios
ventas_por_producto_tienda = sales_df.groupby(['ID_Tienda', 'Producto'])['Cantidad_Vendida'].sum().reset_index()
inventarios_df = inventories_df.merge(ventas_por_producto_tienda, on=['ID_Tienda', 'Producto'], how='left')
inventarios_df['Cantidad_Vendida'].fillna(0, inplace=True)

# Rotación de inventarios = ventas / stock
inventarios_df['Rotacion'] = inventarios_df['Cantidad_Vendida'] / inventarios_df['Stock_Disponible']

# Tiendas con inventario crítico (<10% vendido respecto al stock)
inventarios_df['Porcentaje_Vendido'] = inventarios_df['Cantidad_Vendida'] / inventarios_df['Stock_Disponible']
tiendas_criticas = inventarios_df[inventarios_df['Porcentaje_Vendido'] < 0.10]

6. 😊 Análisis de satisfacción del cliente

In [164]:
# Filtrar tiendas con satisfacción < 60%
tiendas_baja_satisfaccion = satisfaction_df[satisfaction_df['Satisfacción_Promedio'] < 60]

# Recomendación: cruzar con ingresos para ver si hay correlación
satisfaccion_ingresos = satisfaction_df.merge(ingresos_por_tienda.reset_index(), on='ID_Tienda')

7. 🧮 Cálculos con Numpy

In [165]:
# Convertir ingresos totales a array de Numpy
ventas_array = ingresos_por_tienda.to_numpy()

# Mediana y desviación estándar
mediana_ventas = np.median(ventas_array)
desviacion_ventas = np.std(ventas_array)

# Simulación de proyecciones de ventas futuras
np.random.seed(42)
proyecciones = ventas_array + np.random.normal(loc=0, scale=desviacion_ventas, size=len(ventas_array))

📊 Resultados de ventas

In [166]:
# Ventas totales por tienda
print("🔹 Ventas totales por tienda:")
print(ventas_por_tienda)

# Ingresos totales por tienda
print("\n🔹 Ingresos totales por tienda:")
print(ingresos_por_tienda)

# Resumen estadístico de ingresos
print("\n🔹 Resumen estadístico de ingresos:")
print(resumen_ventas)

# Promedio por categoría (si existe)
if 'Categoría' in sales_df.columns:
    print("\n🔹 Promedio de ventas por tienda y categoría:")
    print(promedio_por_categoria)

🔹 Ventas totales por tienda:
ID_Tienda
1    35
2    55
3    50
4    60
5    50
Name: Cantidad_Vendida, dtype: int64

🔹 Ingresos totales por tienda:
ID_Tienda
1     5000
2    10500
3     9000
4    13000
5    13000
Name: Ingreso, dtype: int64

🔹 Resumen estadístico de ingresos:
count       10.000000
mean      5050.000000
std       3361.960407
min       1000.000000
25%       2625.000000
50%       3500.000000
75%       7875.000000
max      10500.000000
Name: Ingreso, dtype: float64


📦 Resultados de inventarios

In [167]:
# Rotación de inventarios por producto y tienda
print("\n🔹 Rotación de inventarios por producto y tienda:")
print(inventarios_df[['ID_Tienda', 'Producto', 'Stock_Disponible', 'Rotacion']])

# Tiendas con inventario crítico
print("\n🔹 Tiendas con inventario crítico (<10% vendido respecto al stock):")
print(tiendas_criticas[['ID_Tienda', 'Producto', 'Porcentaje_Vendido']])


🔹 Rotación de inventarios por producto y tienda:
   ID_Tienda    Producto  Stock_Disponible  Rotacion
0          1  Producto A                50  0.400000
1          1  Producto B                40  0.375000
2          2  Producto A                60  0.500000
3          2  Producto C                45  0.555556
4          3  Producto A                30  0.333333
5          3  Producto B                80  0.500000
6          4  Producto C                70  0.500000
7          4  Producto A                50  0.500000
8          5  Producto B                40  0.500000
9          5  Producto C                60  0.500000

🔹 Tiendas con inventario crítico (<10% vendido respecto al stock):
Empty DataFrame
Columns: [ID_Tienda, Producto, Porcentaje_Vendido]
Index: []


😊 Resultados de satisfacción

In [168]:
# Tiendas con satisfacción menor al 60%
print("\n🔹 Tiendas con baja satisfacción (<60%):")
print(tiendas_baja_satisfaccion)

# Relación entre satisfacción e ingresos
print("\n🔹 Relación entre satisfacción e ingresos:")
print(satisfaccion_ingresos)


🔹 Tiendas con baja satisfacción (<60%):
   ID_Tienda  Satisfacción_Promedio Fecha_Evaluación
4          5                     55       2023-01-15

🔹 Relación entre satisfacción e ingresos:
   ID_Tienda  Satisfacción_Promedio Fecha_Evaluación  Ingreso
0          1                     85       2023-01-15     5000
1          2                     90       2023-01-15    10500
2          3                     70       2023-01-15     9000
3          4                     65       2023-01-15    13000
4          5                     55       2023-01-15    13000


🧮 Resultados con Numpy

In [169]:
# Mediana y desviación estándar
print("\n🔹 Mediana de ingresos por tienda:")
print(mediana_ventas)

print("\n🔹 Desviación estándar de ingresos por tienda:")
print(desviacion_ventas)

# Proyecciones simuladas
print("\n🔹 Proyecciones simuladas de ingresos futuros:")
print(proyecciones)


🔹 Mediana de ingresos por tienda:
10500.0

🔹 Desviación estándar de ingresos por tienda:
2973.213749463701

🔹 Proyecciones simuladas de ingresos futuros:
[ 6476.83734929 10088.9106787  10925.71646685 17528.29330992
 12303.81196679]
